#### PM10 combining

In [ ]:
import pandas as pd
import glob
import os

pm10_files = glob.glob("PM10/*.csv")

df_list = []
for file in pm10_files:
    # Skip empty files
    if os.path.getsize(file) == 0:
        continue

    # Skip files with no header/columns
    with open(file, "r") as f:
        first_line = f.readline().strip()
        if not first_line:  # blank line
            continue

    # Read CSV
    df = pd.read_csv(file)
    df_list.append(df)

# New dataframe
combined_pm10 = pd.concat(df_list, ignore_index=True)
# Drop the end_time column
combined_pm10 = combined_pm10.drop(columns=["end_time"])
# Convert start_time to datetime, keep only the date, and rename to 'date'
combined_pm10["date"] = pd.to_datetime(combined_pm10["start_time"]).dt.date
# Drop the original start_time column
combined_pm10 = combined_pm10.drop(columns=["start_time"])
# Reorder the columns
combined_pm10 = combined_pm10[["city", "station", "date", "value"]]
combined_pm10 = combined_pm10.rename(columns={"value": "pm10"})
combined_pm10


,city,station,start_time,end_time,value
0,Würzburg,539,2016-01-01 11:00:00,2016-01-01 12:00:00,53.0
1,Würzburg,539,2016-01-02 11:00:00,2016-01-02 12:00:00,15.0
2,Würzburg,539,2016-01-03 11:00:00,2016-01-03 12:00:00,29.0
3,Würzburg,539,2016-01-04 11:00:00,2016-01-04 12:00:00,38.0
4,Würzburg,539,2016-01-05 11:00:00,2016-01-05 12:00:00,28.0
...,...,...,...,...,...
7301,Würzburg,511,2025-12-27 11:00:00,2025-12-27 12:00:00,21.0
7302,Würzburg,511,2025-12-28 11:00:00,2025-12-28 12:00:00,19.0
7303,Würzburg,511,2025-12-29 11:00:00,2025-12-29 12:00:00,14.0
7304,Würzburg,511,2025-12-30 11:00:00,2025-12-30 12:00:00,11.0


#### PM2.5 combining

In [38]:
import pandas as pd
import glob
import os

pm25_files = glob.glob("PM25/*.csv")

df_list = []
for file in pm25_files:
    # Skip empty files
    if os.path.getsize(file) == 0:
        continue

    # Skip files with no header/columns
    with open(file, "r") as f:
        first_line = f.readline().strip()
        if not first_line:  # blank line
            continue

    # Read CSV
    df = pd.read_csv(file)
    df_list.append(df)

# New dataframe
combined_pm25 = pd.concat(df_list, ignore_index=True)
# Drop the end_time column
combined_pm25 = combined_pm25.drop(columns=["end_time"])
# Convert start_time to datetime, keep only the date, and rename to 'date'
combined_pm25["date"] = pd.to_datetime(combined_pm25["start_time"]).dt.date
# Drop the original start_time column
combined_pm25 = combined_pm25.drop(columns=["start_time"])
# Reorder the columns
combined_pm25 = combined_pm25[["city", "station", "date", "value"]]
combined_pm25 = combined_pm25.rename(columns={"value": "pm25"})
combined_pm25

,city,station,date,pm25
0,Aachen,1250,2020-01-01,NaN
1,Aachen,1250,2020-01-02,NaN
2,Aachen,1250,2020-01-03,NaN
3,Aachen,1250,2020-01-04,NaN
4,Aachen,1250,2020-01-05,NaN
...,...,...,...,...
359379,Würzburg,511,2025-12-27,15.0
359380,Würzburg,511,2025-12-28,15.0
359381,Würzburg,511,2025-12-29,11.0
359382,Würzburg,511,2025-12-30,9.0


#### Combine data

In [ ]:
# Merge the dataframes
measurements_raw = combined_pm10.merge(
    combined_pm25,
    on=["city", "station", "date"],
    how="outer"
)

# Aggregate the mean for every station in the city
daily_measurements = measurements_raw.groupby(["city", "date"], as_index = False).agg({
    "pm10": "mean",
    "pm25": "mean"
})

# Save the cleaned and combined pollution data
daily_measurements.to_csv("Top100_DailyCityPollution.csv", index = False)